# Phase 6B — Capability suite

This notebook tests **what the system can actually do**, not whether its
plumbing runs.  Every section states a *real* criterion (a quantity the
intended behaviour must satisfy) and an **ablation or baseline** so a green
result means the capability is present, not that a number happened to land.

The system at this phase is a **sensorimotor substrate with intrinsic
motivation and a forward model** — local learning only (STDP, three-factor,
node-perturbation REINFORCE, Marr-Albus), no backprop.  So the capabilities
that exist to be tested are:

| # | Capability | Real criterion | Control |
|---|-----------|----------------|---------|
| C1 | Motor: learns to reach | success/meanD improves over episodes, beats random | frozen-readout ablation must stay flat |
| C2 | Cerebellar forward model | held-out motor-PE drops after babbling | vs untrained init |
| C3 | World model predicts | held-out sensory-PE lower than untrained | vs persistence baseline |
| C4 | Curiosity = learning progress | LP positive while learning, decays on mastery | raw-|PE| does not (Oudeyer) |
| C5 | Cortex L5 encodes the target | L5→(target−tip) linear R² above chance | shuffled-target control |
| C6 | Endogenous sleep dynamics | adenosine/ATP move, phase transitions, no NaN | — |

What is **not** tested because it is **not implemented yet** (Phase 7+):
concepts/semantics (ATL), planning (`expected_free_energy` is an orphan),
language, spatial maps.  Predictive accuracy (C3) is the only sense in which
this system "understands" the world today.

## 0. Install + repo (Colab)

In [ ]:
# Colab only. Skip if running locally with deps already present.
!pip -q install "jax[cuda12]" equinox mujoco mujoco-mjx "imageio[ffmpeg]" numpy

In [ ]:
import os, sys, subprocess
# Clone the repo (idempotent) and work from it.
REPO = 'https://github.com/MateuszSieczka/Neuro_MVP'
DEST = '/content/Neuro_MVP'
if not os.path.isdir(DEST):
    subprocess.check_call(['git', 'clone', '-q', REPO, DEST])
os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)
import jax
print('CWD =', os.getcwd(), '| jax', jax.__version__, '| devices', jax.devices())

## 1. Imports + shared instrumentation

In [ ]:
import time
import numpy as np
import jax, jax.numpy as jnp, equinox as eqx

from core.backend import DEFAULT, DTYPE, make_key
from core.brain_graph import action_brain_cognitive_step
from core.world_model import wm_learning_progress, wm_curiosity_signal
from embodiment.reacher_env import build_reacher
from embodiment.mjx_run_loop import (
    run_babbling, run_reach_episode, _sync_brain_to_body,
)


@eqx.filter_jit
def probe_scan(state, body, sensory, prev_d, params, ctx, keys, learn_readout):
    """Production-faithful babble scan that also emits per-cycle probes.

    Drives the body with M1's command (reward = 0, as in babbling) and
    records the signals the capability tests need.  ``learn_readout``
    gates the M1 reward update exactly as the real babble/reach loops do.
    """
    def step(carry, k):
        st, bd, sn, pd = carry
        kb, kbody = jax.random.split(k)
        out = action_brain_cognitive_step(
            st, params, ctx, sn,
            prev_reward=jnp.asarray(0.0, DTYPE), prev_done=pd, key=kb,
            learn_motor_readout=learn_readout,
        )
        st = out.state
        jc = st.m1.last_joint_command
        pred = st.last_predicted_joint_angles      # forward-model prediction of next q
        bd, samp = bd.act_continuous(kbody, jc)
        st = _sync_brain_to_body(st, bd, jc, params.n_body_actions)
        fwd_err = jnp.mean(jnp.abs(st.last_joint_angles - pred))   # |q_actual_next - q_pred|
        rec = dict(
            fwd_err=fwd_err,
            wm_pe=wm_curiosity_signal(st.world_model, params.world_model),  # pe_short = mean|sensory PE|
            lp=wm_learning_progress(st.world_model, params.world_model),
            l5=out.cortex_l5_rate,
            target=bd.target_xy,
            tip=bd.tip_xy(),
            dist=jnp.linalg.norm(bd.tip_xy() - bd.target_xy),
        )
        return (st, bd, samp.sensory, samp.done), rec
    init = (state, body, sensory, prev_d)
    final, traces = jax.lax.scan(step, init, keys)
    return final, traces


def run_probe(state, body, params, key, n, learn_readout=False):
    keys = jax.random.split(key, n + 1)
    body, samp = body.reset(keys[0])
    final, tr = probe_scan(
        state, body, samp.sensory, samp.done, params, DEFAULT, keys[1:], learn_readout,
    )
    jax.block_until_ready(tr['fwd_err'])
    return final[0], final[1], {k: np.asarray(v) for k, v in tr.items()}


def random_baseline_meanD(body, key, n_ep=10, ep_len=200):
    md_ = body.cfg.motor_dim
    means = []
    for ep in range(n_ep):
        key, kr = jax.random.split(key)
        b, _ = body.reset(kr)
        ds = []
        for _ in range(ep_len):
            key, k1, k2 = jax.random.split(key, 3)
            jc = jax.random.uniform(k1, (md_,), DTYPE, -1.0, 1.0)
            b, _ = b.act_continuous(k2, jc)
            ds.append(float(jnp.linalg.norm(b.tip_xy() - b.target_xy)))
        means.append(float(np.mean(ds[-50:])))
    return float(np.mean(means))

verdicts = {}
print('instrumentation ready')

## C1 — Motor: does it actually LEARN to reach?

**Criterion:** after babbling (forward-model warm-up) the reach success rate /
mean tip→target distance improves over episodes and beats a random policy.

**Ablation:** an identical run with the M1 readout **frozen**
(`m1_readout_lr=0`) must *not* improve.  If the learning run improves and the
frozen run stays flat, the improvement is caused by readout learning — not by
geometry, the cerebellum alone, or luck.

In [ ]:
N_BABBLE = 1500
N_EP = 24
EP_LEN = 200

def reach_curve(m1_lr, tag):
    params, state, body = build_reacher(make_key(0), m1_readout_lr=m1_lr)
    bab = run_babbling(state, params, DEFAULT, body, make_key(1),
                       n_cycles=N_BABBLE, target_refresh=300)
    state, body = bab.brain_state, bab.body
    succ, meanD = [], []
    key = make_key(2)
    for ep in range(N_EP):
        key, k = jax.random.split(key)
        res = run_reach_episode(state, params, DEFAULT, body, k,
                                max_steps=EP_LEN, reset_body=True)
        state, body = res.brain_state, res.body
        succ.append(float(res.success))
        meanD.append(float(jnp.mean(res.dists[-50:])))
    print(f'[{tag}] first5 meanD={np.mean(meanD[:5]):.3f}  last5 meanD={np.mean(meanD[-5:]):.3f}  '
          f'success_rate(last10)={np.mean(succ[-10:]):.2f}')
    return np.array(meanD), np.array(succ), body

learn_meanD, learn_succ, body_l = reach_curve(5e-3, 'LEARN')
froz_meanD,  froz_succ,  _      = reach_curve(0.0,  'FROZEN')
rand_meanD = random_baseline_meanD(body_l, make_key(99))
print(f'random meanD = {rand_meanD:.3f}')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(learn_meanD, label='learn'); ax[0].plot(froz_meanD, label='frozen readout')
ax[0].axhline(rand_meanD, c='k', ls='--', label='random'); ax[0].set_title('mean tip→target (last 50 steps)'); ax[0].legend(); ax[0].set_xlabel('episode')
ax[1].plot(np.cumsum(learn_succ)/(np.arange(N_EP)+1), label='learn')
ax[1].plot(np.cumsum(froz_succ)/(np.arange(N_EP)+1), label='frozen'); ax[1].set_title('cumulative success rate'); ax[1].legend(); ax[1].set_xlabel('episode')
plt.tight_layout(); plt.show()

learn_improve = np.mean(learn_meanD[:5]) - np.mean(learn_meanD[-5:])   # >0 means got closer
froz_improve  = np.mean(froz_meanD[:5])  - np.mean(froz_meanD[-5:])
beats_random  = np.mean(learn_meanD[-5:]) < rand_meanD
print(f'learn_improve={learn_improve:+.3f}  froz_improve={froz_improve:+.3f}  beats_random={beats_random}')
verdicts['C1 motor learns reach'] = bool(
    learn_improve > 0.02 and learn_improve > froz_improve + 0.01 and beats_random
)

## C2 — Cerebellar forward model: does prediction error drop?

The cerebellum (adaptive filter; Dean & Porrill 2010) predicts the next joint
angle; with the loop closed (`predicted = jc + cb_forward`) babbling should
shrink the **held-out** forward-model error.

**Criterion:** mean `|q_actual − q_predicted|` on an unseen babble seed after
babbling ≤ 70 % of the same measured on the untrained brain.

In [ ]:
params, state0, body0 = build_reacher(make_key(0))

# Untrained baseline on a held-out seed.
_, _, tr_init = run_probe(state0, body0, params, make_key(777), n=300, learn_readout=False)
# Train the forward model with babbling, then re-measure on the SAME held-out seed.
bab = run_babbling(state0, params, DEFAULT, body0, make_key(1), n_cycles=2000, target_refresh=400)
_, _, tr_trained = run_probe(bab.brain_state, bab.body, params, make_key(777), n=300, learn_readout=False)

fe_init = float(tr_init['fwd_err'][50:].mean())
fe_trained = float(tr_trained['fwd_err'][50:].mean())
ratio = fe_trained / (fe_init + 1e-9)
print(f'held-out forward-PE: init={fe_init:.4f}  trained={fe_trained:.4f}  ratio={ratio:.2f}')
verdicts['C2 cerebellum forward model'] = bool(ratio <= 0.7)

## C3 — World model: does it predict the world better than baselines?

This is the only sense in which the system currently "understands" the world:
predictive accuracy.  `wm_pe` is the smoothed mean sensory prediction error.

**Criterion:** held-out `wm_pe` after self-supervised learning is below both
(a) the untrained brain and (b) a persistence baseline (predict next = current
sensory).

In [ ]:
params, s0, b0 = build_reacher(make_key(0))
_, _, tr_i = run_probe(s0, b0, params, make_key(888), n=300, learn_readout=False)
bab = run_babbling(s0, params, DEFAULT, b0, make_key(3), n_cycles=2500, target_refresh=400)
_, _, tr_t = run_probe(bab.brain_state, bab.body, params, make_key(888), n=300, learn_readout=False)

wm_init = float(tr_i['wm_pe'][100:].mean())
wm_trained = float(tr_t['wm_pe'][100:].mean())

# Persistence baseline: ||sensory_{t+1} - sensory_t|| on the held-out seed, in
# the same per-dim mean-abs units as wm_pe.
keys = jax.random.split(make_key(888), 301)
b, samp = b0.reset(keys[0]); sn_prev = samp.sensory
deltas = []
for k in keys[1:]:
    k1, k2 = jax.random.split(k)
    jc = jax.random.uniform(k1, (b0.cfg.motor_dim,), DTYPE, -1, 1)  # match exploration regime
    b, samp = b.act_continuous(k2, jc)
    deltas.append(float(jnp.mean(jnp.abs(samp.sensory - sn_prev)))); sn_prev = samp.sensory
persist = float(np.mean(deltas[100:]))
print(f'held-out wm_pe: init={wm_init:.4f}  trained={wm_trained:.4f}  persistence={persist:.4f}')
verdicts['C3 world model predicts'] = bool(wm_trained < wm_init and wm_trained < persist)

## C4 — Curiosity is learning-progress, not raw surprise (Oudeyer 2007)

**Criterion:** during babbling the learning-progress signal `lp = pe_long −
pe_short` is **positive** while the world model is improving and **decays
toward 0** as the region is mastered, while raw surprise `wm_pe` falls.  This is
the IAC signature that drives the agent to the learning frontier and is immune
to the noisy-TV trap (raw |PE| would stay high on irreducible noise).

In [ ]:
params, s0, b0 = build_reacher(make_key(0))
_, _, tr = run_probe(s0, b0, params, make_key(5), n=2500, learn_readout=False)
lp = tr['lp']; wm_pe = tr['wm_pe']
early_lp = float(lp[100:600].mean()); late_lp = float(lp[-500:].mean())
print(f'wm_pe: start={wm_pe[:200].mean():.4f} -> end={wm_pe[-200:].mean():.4f}')
print(f'learning progress: early={early_lp:+.4f}  late={late_lp:+.4f}')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(wm_pe); ax[0].set_title('wm_pe (raw surprise, pe_short)'); ax[0].set_xlabel('cycle')
ax[1].plot(lp); ax[1].axhline(0, c='k', ls=':'); ax[1].set_title('learning progress (pe_long - pe_short)'); ax[1].set_xlabel('cycle')
plt.tight_layout(); plt.show()
# Learning happened (surprise dropped) and LP was positive while it did, smaller once mastered.
verdicts['C4 curiosity = learning progress'] = bool(
    wm_pe[:200].mean() > wm_pe[-200:].mean() and early_lp > 0 and early_lp > late_lp
)

## C5 — Cortex L5 encodes the target (necessary condition for reaching)

M1 reads cortex L5.  For reaching to be *possible at all*, L5 must carry
target-relative information.  We fit a linear map L5 → (target − tip) and report
R²; a **shuffled-target** control gives the chance level.

**Criterion:** real R² is clearly above the shuffled control.

In [ ]:
params, s0, b0 = build_reacher(make_key(0))
bab = run_babbling(s0, params, DEFAULT, b0, make_key(1), n_cycles=800, target_refresh=200)
_, _, tr = run_probe(bab.brain_state, bab.body, params, make_key(11), n=600, learn_readout=False)
L5 = tr['l5']                       # (N, n_l5)
Y = (tr['target'] - tr['tip'])      # (N, 2)
L5 = L5[50:]; Y = Y[50:]

def lin_r2(X, Y):
    X1 = np.concatenate([X, np.ones((X.shape[0], 1))], 1)
    W, *_ = np.linalg.lstsq(X1, Y, rcond=None)
    pred = X1 @ W
    ss_res = ((Y - pred) ** 2).sum(); ss_tot = ((Y - Y.mean(0)) ** 2).sum() + 1e-9
    return 1 - ss_res / ss_tot

r2 = lin_r2(L5, Y)
rng = np.random.default_rng(0)
r2_shuf = np.mean([lin_r2(L5, Y[rng.permutation(len(Y))]) for _ in range(5)])
print(f'L5 -> (target-tip)  R2={r2:.3f}   shuffled-control R2={r2_shuf:.3f}')
verdicts['C5 L5 encodes target'] = bool(r2 > 0.15 and r2 > r2_shuf + 0.1)

## C6 — Endogenous sleep dynamics

The adenosine Process-S integrator should move ATP / sleep pressure over a long
run and the brain must stay finite.  (The full "sleep improves reach" claim
needs hundreds of GPU episodes — out of scope here; this checks the homeostat
is *alive and endogenous*, not stuck.)

In [ ]:
params, state, body = build_reacher(make_key(0))
key = make_key(1)
aden0 = float(state.neuromodulator.adenosine)
atp0 = float(jnp.mean(state.astrocyte.atp))
phases = []
for ep in range(6):
    key, k = jax.random.split(key)
    res = run_reach_episode(state, params, DEFAULT, body, k, max_steps=300, reset_body=True)
    state, body = res.brain_state, res.body
    phases.append(int(state.sleep.phase))
aden1 = float(state.neuromodulator.adenosine)
atp1 = float(jnp.mean(state.astrocyte.atp))
finite = bool(jnp.all(jnp.isfinite(state.astrocyte.atp)) and jnp.all(jnp.isfinite(res.tip_traj)))
print(f'adenosine {aden0:.4f}->{aden1:.4f}  atp {atp0:.4f}->{atp1:.4f}  phases={phases}  finite={finite}')
verdicts['C6 sleep homeostat alive'] = bool(finite and (abs(aden1 - aden0) > 1e-6 or abs(atp1 - atp0) > 1e-6))

## Decision matrix

In [ ]:
print('=== CAPABILITY VERDICTS ===')
for k, v in verdicts.items():
    print(f'  [{"PASS" if v else "FAIL"}]  {k}')
print()
print('PASS = capability demonstrated with its control/baseline.')
print('FAIL = re-read the section numbers above; the binding constraint is')
print('       usually upstream (C5 then C2 then C1).')